# 00 · Setup check

Run the cell below (click it, then press **Shift + Enter**).

- ✅ means that part is ready.
- ❌ comes with a fix underneath. Do the fix, then run the cell again.
- Still red after two tries? Ask your setup buddy first, then put your hand up.

This checks the setup you will keep using in your programme modules, not just today.

In [ ]:
import sys, shutil, subprocess, importlib, os
from pathlib import Path

ok_count, problems = 0, []

def ok(msg):
    global ok_count
    ok_count += 1
    print("✅", msg)

def bad(msg, fix):
    problems.append(msg)
    print("❌", msg)
    print("    FIX:", fix)

def git(*args):
    try:
        r = subprocess.run(["git", *args], capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else ""
    except Exception:
        return ""

# --- Where is this running? ---------------------------------------------------
if "google.colab" in sys.modules:
    where = "Google Colab (fine for today, but set up your own laptop as well)"
elif os.environ.get("CODESPACES"):
    where = "GitHub Codespaces (browser)"
elif Path(sys.prefix, "conda-meta").exists():
    where = "Anaconda / conda on this laptop"
else:
    where = "Python on this laptop"
print("Running in:", where)
print("Python used by this notebook:", sys.executable)
print()

# Homebrew / system Pythons refuse 'pip install' unless you use a project environment (.venv)
import sysconfig
in_env = sys.prefix != sys.base_prefix or Path(sys.prefix, "conda-meta").exists()
apple_python = sys.platform == "darwin" and (sys.prefix.startswith("/Library/Developer/CommandLineTools")
                                            or sys.prefix.startswith("/Applications/Xcode")
                                            or sys.prefix.startswith("/System/"))
locked = (Path(sysconfig.get_paths()["stdlib"], "EXTERNALLY-MANAGED").exists() or apple_python) and not in_env
VENV_FIX = ("this is a Homebrew/Apple system Python. Don't install packages into it. Make a project environment:\n"
            "         Cmd/Ctrl+Shift+P -> 'Python: Create Environment' -> Venv -> tick requirements.txt,\n"
            "         then Select Kernel -> Python Environments -> .venv, and run this cell again.")
if locked:
    print("⚠️  Note:", VENV_FIX)
    print()

# --- Python version -----------------------------------------------------------
v = sys.version_info
if v >= (3, 10):
    ok(f"Python {v.major}.{v.minor}")
else:
    bad(f"Python {v.major}.{v.minor} is too old for current pandas",
        "pick a newer Python as the kernel (Anaconda, or a .venv made from a newer Python - see SETUP.md).")

# --- Packages -----------------------------------------------------------------
needed = {
    "pandas": "tables of data (DataFrames)",
    "numpy": "fast maths on arrays",
    "matplotlib": "charts",
    "openpyxl": "reading/writing Excel files",
}
optional = {"yfinance": "downloading stock prices (optional notebook)"}

for name, why in {**needed, **optional}.items():
    try:
        m = importlib.import_module(name)
        ok(f"{name} {getattr(m, '__version__', '')} - {why}")
    except ImportError:
        tag = " (optional)" if name in optional else ""
        bad(f"{name} is missing{tag} - {why}",
            VENV_FIX if locked else
            f"run  %pip install {name}  in a new cell below, then Restart the kernel and run this cell again.")

# --- Git ----------------------------------------------------------------------
print()
if shutil.which("git"):
    ok(git("--version"))
    name, email = git("config", "--global", "user.name"), git("config", "--global", "user.email")
    if name and email:
        ok(f"Git knows who you are: {name} <{email}>")
    else:
        bad("Git doesn't know your name/email yet",
            'in the VS Code terminal run:\n'
            '         git config --global user.name "Your Name"\n'
            '         git config --global user.email "the-email-you-used-for-GitHub"')
    if git("config", "--global", "pull.rebase"):
        ok("Git pull setting is configured")
    else:
        bad("Git pull setting not set (needed for the pair exercise)",
            "in the VS Code terminal run:  git config --global pull.rebase false")
    if git("rev-parse", "--is-inside-work-tree") == "true":
        remote = git("remote", "get-url", "origin")
        ok(f"This folder is a Git repository (remote: {remote or 'none yet'})")
    else:
        bad("This folder is not a Git repository yet",
            "that's expected if you downloaded a ZIP. Follow the Git steps in README.md to clone your own copy.")
else:
    bad("Git is not installed (or VS Code can't find it)",
        "Windows: install from https://git-scm.com/download/win (keep the default options), then fully close and reopen VS Code.\n"
        "         Mac: open Terminal and run  xcode-select --install , then reopen VS Code.")

# --- Summary ------------------------------------------------------------------
print()
if not problems:
    print("🎉 All green. You're ready. Go and help your neighbour!")
else:
    print(f"{len(problems)} thing(s) to fix:", *problems, sep="\n  - ")

### Need to install a missing package?

Type it in the cell below and run it, for example `%pip install yfinance`.
Using `%pip` inside the notebook installs into **the same Python this notebook is running**, which avoids the most common "I installed it but it still says missing" problem.
After installing: **Restart** the kernel (button at the top of the notebook), then run the check again.

In [ ]:
# %pip install yfinance